In [2]:
text_list = []
with open('./data/jokes.txt', 'r') as f:
    for line in f:
        text_list.append(str(line))

In [3]:
import yaml
with open("configs/config.yaml", 'r') as f:
    config = yaml.safe_load(f)

In [4]:
from TelegramJokes.models import JokesTokenizer

In [5]:
tokenizer = JokesTokenizer(config['model']['model_params']['vocab_size'], config['model']['tokenizer_params']['special_tokens'])  
tokenizer.load("artifacts/checkpoints/tokenizers/tokenizer_lstm_v01.json")

In [396]:
#from TelegramJokes.models import LSTMModel
import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
class LSTMModel(nn.Module):
    def __init__(self, vocab_size, embedding_size, hidden_size, num_layers, dropout, **kwargs):
        super(LSTMModel, self).__init__()
        self.embedding_size = embedding_size
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.dropout = dropout
        self.vocab_size = vocab_size

        self.embeddings = nn.Embedding(
            num_embeddings=self.vocab_size,
            embedding_dim=self.embedding_size
        )
        self.encoder = nn.LSTM(
            input_size=self.embedding_size,
            hidden_size=self.hidden_size,
            num_layers=self.num_layers,
            batch_first=True,
            dropout=self.dropout
            )

        self.head = nn.Linear(hidden_size, self.vocab_size)

    def forward(self, x):
        emb = self.embeddings(x)
        out, (h, c) = self.encoder(emb)
        out = self.head(out)
        return out

In [29]:
def generate(self, input_text):
        input_tokens = self.tokenizer.encode(input_text)

        input_tokens = torch.tensor(input_tokens[:-1], dtype=torch.long).unsqueeze(0).to(self.device)

        temperature = 0.5
        for i in range(100):
            self._model.eval()
            model_out = self._model(input_tokens).squeeze(0)
            model_p = torch.softmax(model_out/temperature, 1)
            sample_tokens = torch.multinomial(model_p[-1], 1)
            #next_token = model(input_tokens).argmax(-1).squeeze(0)[-1]
            input_tokens = torch.concat((input_tokens, sample_tokens.unsqueeze(0)), 1)
            if input_tokens[0,-1]==3:
                break

        output_tokens = input_tokens.squeeze(0).cpu()
        output_text = self.tokenizer.decode(list(output_tokens))
        return output_text

In [397]:
def generate_1(model, tokenizer, prompt, maxlen, temperature):
    tokens = tokenizer.encode(prompt)
    input = torch.tensor(tokens[:-1], dtype=torch.long).unsqueeze(0)

    h, c = None, None
    for i in range(maxlen):
        emb = model.embeddings(input)
        out, (h,c) = model.encoder(emb)
        #if h is None:
        #    out, (h,c) = model.encoder(emb)
        #else:
        #    out, (h,c) = model.encoder(emb, (h, c))
        logits = model.head(out) / temperature
        next_token = logits.argmax(-1)[:,-1].unsqueeze(0)
        input = torch.cat((input, next_token), -1)
        if input[:,-1]==3:
            break
    output_text = tokenizer.decode(list(input.squeeze(0)))
    return output_text

def generate_2(model, tokenizer, prompt, maxlen, temperature):
    tokens = tokenizer.encode(prompt)
    generated = tokens[:-1]
    input = torch.tensor(generated, dtype=torch.long).unsqueeze(0)

    h, c = None, None
    for i in range(maxlen):
        emb = model.embeddings(input)

        if h is None:
           out, (h,c) = model.encoder(emb)
        else:
           out, (h,c) = model.encoder(emb, (h, c))
        
        logits = model.head(out)[:,-1] / temperature
        input = logits.argmax(-1).unsqueeze(0)
        generated.append(input.item())

        if generated[-1]==3:
            break
    output_text = tokenizer.decode(generated)
    return output_text

def generate_3(model, tokenizer, prompt, maxlen, temperature):
    tokens = tokenizer.encode(prompt)
    input = torch.tensor(tokens[:-1], dtype=torch.long).unsqueeze(0)

    h, c = None, None
    for i in range(maxlen):
        emb = model.embeddings(input)
        out, (h,c) = model.encoder(emb)
        logits = model.head(out) / temperature
        probs = torch.softmax(logits, -1)
        next_token = torch.multinomial(probs[:,-1], 1)
        input = torch.cat((input, next_token), -1)
        if input[:,-1]==3:
            break
    output_text = tokenizer.decode(list(input.squeeze(0)))
    return output_text

def generate_4(model, tokenizer, prompt, maxlen, temperature):
    tokens = tokenizer.encode(prompt)
    generated = tokens[:-1]
    input = torch.tensor(generated, dtype=torch.long).unsqueeze(0)

    h, c = None, None
    for i in range(maxlen):
        emb = model.embeddings(input)

        if h is None:
           out, (h,c) = model.encoder(emb)
        else:
           out, (h,c) = model.encoder(emb, (h, c))
        
        logits = model.head(out)[:,-1] / temperature
        probs = torch.softmax(logits, -1)
        input = torch.multinomial(probs[-1], 1).unsqueeze(0)
        generated.append(input.item())
        if generated[-1]==3:
            break
    output_text = tokenizer.decode(generated)
    return output_text

def generate_5(model, tokenizer, prompt, maxlen, temperature=1.0, k=3, length_penalty=0.7, device="cpu"):
    model.eval()

    tokens = tokenizer.encode(prompt)
    generated = tokens[:-1]

    eos = 3

    # beam: (sequence, logprob, finished)
    beams = [(generated, 0.0, False)]

    for step in range(maxlen):

        all_candidates = []

        for seq, score, finished in beams:

            # если последовательность уже завершена — просто переносим её дальше
            if finished:
                all_candidates.append((seq, score, True))
                continue

            inp = torch.tensor(seq, dtype=torch.long, device=device).unsqueeze(0)

            with torch.no_grad():
                logits = model(inp)[:, -1, :] / temperature
                log_probs = F.log_softmax(logits, dim=-1)

            topk_log_probs, topk_ids = torch.topk(log_probs, k)

            for i in range(k):
                token = topk_ids[0, i].item()
                token_logprob = topk_log_probs[0, i].item()

                new_seq = seq + [token]
                new_score = score + token_logprob
                new_finished = token == eos

                all_candidates.append((new_seq, new_score, new_finished))

        # length penalty
        def score_with_penalty(candidate):
            seq, score, _ = candidate
            lp = ((5 + len(seq)) / 6) ** length_penalty
            return score / lp

        # выбираем top-k
        ordered = sorted(all_candidates, key=score_with_penalty, reverse=True)
        beams = ordered[:k]

        # если все последовательности закончились — можно остановиться
        if all(f for (_, _, f) in beams):
            break

    best_seq = beams[0][0]

    return tokenizer.decode(best_seq)

In [398]:
model = LSTMModel(**config['model']['model_params'])
model.eval()

LSTMModel(
  (embeddings): Embedding(5000, 256)
  (encoder): LSTM(256, 256, num_layers=2, batch_first=True, dropout=0.3)
  (head): Linear(in_features=256, out_features=5000, bias=True)
)

In [399]:
import torch
model_state = torch.load("artifacts/checkpoints/models/lstm_v01.pt", map_location=torch.device('cpu'), weights_only=False)

In [400]:
model.load_state_dict(model_state['model_state_dict'])

<All keys matched successfully>

In [401]:
text = "Заходит улитка в бар, "
tokens = tokenizer.encode(text)
input = torch.tensor(tokens, dtype=torch.long).unsqueeze(0)
print(input)

tensor([[   2, 4166, 1656,  127,  195,  108, 1352,    5,   52,    3]])


In [ ]:
%%timeit
generate_1(model, tokenizer, text, 128, 1.0)

43 ms ± 1.97 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [471]:
%%timeit
generate_2(model, tokenizer, text, 128, 1.0)

17.1 ms ± 1.12 ms per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [448]:
%%timeit
generate_3(model.half(), tokenizer, text, 128, 1.0)

The slowest run took 6.16 times longer than the fastest. This could mean that an intermediate result is being cached.
270 ms ± 144 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [449]:
%%timeit
generate_4(model.half(), tokenizer, text, 128, 1.0)

64.6 ms ± 3.57 ms per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [450]:
%%timeit
generate_5(model.half(), tokenizer, text, 128, 1.0)

279 ms ± 40.3 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [454]:
generate_1(model.half(), tokenizer, text, 128, 1.0)

' Заходит улитка в бар,  говорит- У меня есть два билета, а я - вторая.\n'

In [456]:
generate_2(model.half(), tokenizer, text, 128, 1.0)

' Заходит улитка в бар,  говорит- У меня есть два билета, а я - вторая.\n'

In [462]:
generate_3(model.half(), tokenizer, text, 128, 0.7)

' Заходит улитка в бар,  говорит ему- Я хочу две бутылки водки, а я не мог понять, сколько у тебя были восьмого марта...\n'

In [445]:
generate_4(model, tokenizer, "", 128, 0.7)

' - Алло, это платье?- Да!- А что вам привезти?- Свинья.- А ваш ребенок?- Да, популя, спортсмен, сэр.\n'

In [467]:
generate_5(model.half(), tokenizer, "", 128, 0.7)

' - А ты знаешь, что у нас в стране есть?- Есть.- А почему?- Потому что они не работают.\n'